# Credit card default — EDA + RF + XGBoost

Need `xgboost` in your environment (conda usually works; on some Macs you may need OpenMP / `libomp` for pip installs).


In [12]:
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, recall_score
from xgboost import XGBClassifier

df = pd.read_csv("creditcard.csv")
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,400000,1,1,2,32,0,0,0,0,0,...,48272,49478,51242,3028,3023,3000,3000,3000,38662,0
1,120000,2,2,2,30,-1,-1,-1,-1,-1,...,1964,1883,1538,3230,3011,1964,1883,1538,1911,0
2,270000,2,2,2,32,0,0,0,0,0,...,94856,86461,83650,1808,69563,2891,2689,3012,2771,0
3,280000,2,2,1,27,0,0,0,0,0,...,257689,193231,191143,11052,9563,15017,5374,5420,6021,0
4,30000,2,1,2,27,0,0,-1,0,0,...,1814,0,0,1000,664,1500,0,0,0,0


In [6]:
#report = sv.analyze(df, target_feat="default payment next month")
#report.show_html("creditcard_sweetviz.html")

In [7]:
print(df.shape)
print(df["default payment next month"].value_counts())
print("missing", df.isnull().sum().sum())
df.describe()


(24000, 24)
default payment next month
0    18691
1     5309
Name: count, dtype: int64
missing 0


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
count,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000,...,24000.000000,24000.000000,24000.000000,24000.000000,2.400000e+04,24000.000000,24000.000000,24000.000000,24000.000000,24000.000000
mean,167091.653333,1.606708,1.852042,1.551083,35.502833,-0.024500,-0.140958,-0.170125,-0.226125,-0.270583,...,43073.483917,40199.636167,38829.113250,5657.188875,5.908712e+03,5222.014208,4791.816500,4789.428667,5148.400542,0.221208
std,129220.852530,0.488491,0.786877,0.522400,9.250053,1.120804,1.191315,1.194184,1.166854,1.133911,...,63980.957358,60706.387390,59507.640696,16659.502392,2.420729e+04,17908.072069,15752.428869,15321.277386,17693.315187,0.415069
min,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2261.500000,1721.250000,1209.250000,1000.000000,8.200000e+02,390.000000,296.000000,234.750000,111.750000,0.000000
50%,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,19005.000000,18071.000000,17008.500000,2120.000000,2.010500e+03,1800.000000,1500.000000,1500.000000,1500.000000,0.000000
75%,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,54599.500000,50283.000000,49327.000000,5006.000000,5.000000e+03,4515.250000,4003.250000,4002.250000,4000.000000,0.000000
max,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.000000,621000.000000,426529.000000,528666.000000,1.000000


## EDA highlights

SweetViz writes `creditcard_sweetviz.html` in this folder—open it in a browser for distributions, correlations vs default, and missingness.

Roughly 24k rows, 23 numeric inputs, target `default payment next month`. No missing values in this file. The target is imbalanced (more 0s than 1s), so I use ROC-AUC instead of accuracy. Bill and pay amount columns have wide ranges; payment status fields are useful for trees without a lot of preprocessing.


In [8]:
target = "default payment next month"
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=101, stratify=y
)

# stratified folds so each CV fold keeps similar default rate (imbalanced target)
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestClassifier(random_state=42)
xgb = XGBClassifier(random_state=42, verbosity=0)

rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

print("RF  train ROC-AUC:", round(roc_auc_score(y_train, rf.predict_proba(X_train)[:, 1]), 4))
print("RF  test ROC-AUC: ", round(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), 4))
print("XGB train ROC-AUC:", round(roc_auc_score(y_train, xgb.predict_proba(X_train)[:, 1]), 4))
print("XGB test ROC-AUC: ", round(roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]), 4))

print("\nRF test classification_report")
print(classification_report(y_test, rf.predict(X_test)))
print("XGB test classification_report")
print(classification_report(y_test, xgb.predict(X_test)))

# 5-fold CV on training split only (test set stays untouched)
rf_cv = cross_val_score(rf, X_train, y_train, cv=stratified_cv, scoring="roc_auc")
xgb_cv = cross_val_score(xgb, X_train, y_train, cv=stratified_cv, scoring="roc_auc")

print("RF  CV ROC-AUC mean:", round(rf_cv.mean(), 4), "std:", round(rf_cv.std(), 4))
print("XGB CV ROC-AUC mean:", round(xgb_cv.mean(), 4), "std:", round(xgb_cv.std(), 4))


RF  train ROC-AUC: 1.0
RF  test ROC-AUC:  0.7767
XGB train ROC-AUC: 0.9558
XGB test ROC-AUC:  0.7676

RF test classification_report
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.65      0.38      0.48      1062

    accuracy                           0.82      4800
   macro avg       0.75      0.66      0.68      4800
weighted avg       0.80      0.82      0.80      4800

XGB test classification_report
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      3738
           1       0.64      0.38      0.48      1062

    accuracy                           0.81      4800
   macro avg       0.74      0.66      0.68      4800
weighted avg       0.80      0.81      0.80      4800

RF  CV ROC-AUC mean: 0.7567 std: 0.0112
XGB CV ROC-AUC mean: 0.7528 std: 0.0143


## Models — quick read

I held out 20% as test (`stratify=y` keeps default rate similar in train vs test). **Train** ROC-AUC is on rows the model saw; **test** ROC-AUC is the honest holdout score.

5-fold **StratifiedKFold** CV is run on **training data only** (each fold keeps a similar default rate). The test set is never used in CV. The mean ROC-AUC is an estimate of how the model does on new data; std across folds measures how noisy that estimate is. Compare models with CV mean and check test ROC-AUC last.


## Recall as primary metric

Imbalanced data: lots of non-defaults. **Recall** on class **1** (default) is how many real defaults we **catch**. Below uses the same **StratifiedKFold** and the same fitted `rf` / `xgb` from above—run the modeling cell first.

In [9]:
from sklearn.metrics import recall_score

print("Recall for class 1 (default)")
rf_rec_cv = cross_val_score(rf, X_train, y_train, cv=stratified_cv, scoring="recall")
xgb_rec_cv = cross_val_score(xgb, X_train, y_train, cv=stratified_cv, scoring="recall")
print("RF  CV recall mean:", round(rf_rec_cv.mean(), 4), "std:", round(rf_rec_cv.std(), 4))
print("XGB CV recall mean:", round(xgb_rec_cv.mean(), 4), "std:", round(xgb_rec_cv.std(), 4))
print("RF  train recall:", round(recall_score(y_train, rf.predict(X_train)), 4))
print("RF  test recall: ", round(recall_score(y_test, rf.predict(X_test)), 4))
print("XGB train recall:", round(recall_score(y_train, xgb.predict(X_train)), 4))
print("XGB test recall: ", round(recall_score(y_test, xgb.predict(X_test)), 4))

Recall for class 1 (default)
RF  CV recall mean: 0.3657 std: 0.0221
XGB CV recall mean: 0.3647 std: 0.0243
RF  train recall: 0.9993
RF  test recall:  0.3795
XGB train recall: 0.6
XGB test recall:  0.3795


## Full classification reports (train + test)

Same fitted models as above. Shows precision, recall, F1, and support per class.

In [10]:
labels = [0, 1]
target_names = ["no_default (0)", "default (1)"]

print("=== Random Forest — train ===")
print(classification_report(y_train, rf.predict(X_train), labels=labels, target_names=target_names, digits=4))
print("=== Random Forest — test ===")
print(classification_report(y_test, rf.predict(X_test), labels=labels, target_names=target_names, digits=4))

print("=== XGBoost — train ===")
print(classification_report(y_train, xgb.predict(X_train), labels=labels, target_names=target_names, digits=4))
print("=== XGBoost — test ===")
print(classification_report(y_test, xgb.predict(X_test), labels=labels, target_names=target_names, digits=4))

=== Random Forest — train ===
                precision    recall  f1-score   support

no_default (0)     0.9998    0.9998    0.9998     14953
   default (1)     0.9993    0.9993    0.9993      4247

      accuracy                         0.9997     19200
     macro avg     0.9995    0.9995    0.9995     19200
  weighted avg     0.9997    0.9997    0.9997     19200

=== Random Forest — test ===
                precision    recall  f1-score   support

no_default (0)     0.8423    0.9417    0.8892      3738
   default (1)     0.6490    0.3795    0.4789      1062

      accuracy                         0.8173      4800
     macro avg     0.7456    0.6606    0.6841      4800
  weighted avg     0.7995    0.8173    0.7984      4800

=== XGBoost — train ===
                precision    recall  f1-score   support

no_default (0)     0.8962    0.9813    0.9369     14953
   default (1)     0.9013    0.6000    0.7204      4247

      accuracy                         0.8970     19200
     macro av

## Brief tuning (2–3 knobs per model)

Grid search with **stratified 5-fold** and **`scoring="recall"`** (default class). **RF:** `max_depth`, `min_samples_leaf`, **`class_weight`** (`None` vs `"balanced"`) for imbalance. **XGB:** `max_depth`, `learning_rate`, **`subsample`** (row sampling — regularization). Then compare **test recall** and **test ROC-AUC** to the default models fit earlier (same `random_state=42`).

In [13]:
rf_grid = {
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 2],
    "class_weight": [None, "balanced"],
}
rf_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=1),
    rf_grid,
    cv=stratified_cv,
    scoring="recall",
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)
rf_tuned = rf_search.best_estimator_

xgb_grid = {
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
}
xgb_search = GridSearchCV(
    XGBClassifier(random_state=42, verbosity=0, n_jobs=1),
    xgb_grid,
    cv=stratified_cv,
    scoring="recall",
    n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
xgb_tuned = xgb_search.best_estimator_

print("_RF_")
print("best params", rf_search.best_params_)
print("default test recall", round(recall_score(y_test, rf.predict(X_test)), 4),
      "tuned", round(recall_score(y_test, rf_tuned.predict(X_test)), 4))
print("default test roc_auc ", round(roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), 4),
      "tuned", round(roc_auc_score(y_test, rf_tuned.predict_proba(X_test)[:, 1]), 4))

print("_XGB_")
print("best params", xgb_search.best_params_)
print("default test recall", round(recall_score(y_test, xgb.predict(X_test)), 4),
      "tuned", round(recall_score(y_test, xgb_tuned.predict(X_test)), 4))
print("default test roc_auc ", round(roc_auc_score(y_test, xgb.predict_proba(X_test)[:, 1]), 4),
      "tuned", round(roc_auc_score(y_test, xgb_tuned.predict_proba(X_test)[:, 1]), 4))

_RF_
best params {'class_weight': 'balanced', 'max_depth': 10, 'min_samples_leaf': 2}
default test recall 0.3795 tuned 0.5734
default test roc_auc  0.7767 tuned 0.7923
_XGB_
best params {'learning_rate': 0.1, 'max_depth': 6, 'subsample': 0.8}
default test recall 0.3795 tuned 0.3757
default test roc_auc  0.7676 tuned 0.7924


### Tuning — did it help? Good knobs?

**RF:** I tuned depth/leaf size plus **`class_weight`**. Balanced weights push the model to care more about the minority (default) class, which usually **raises recall** but can **hurt precision or ROC-AUC** if it over-predicts defaults. If test **recall** went **up** but AUC **down**, that matches the tradeoff we asked for when we optimized recall in the grid.

**XGB:** I used **`subsample`** instead of e.g. `scale_pos_weight` this time — subsample often **reduces overfit**; POS weight would directly target imbalance. If recall **improved**, the grid found a better bias–variance spot; if **not**, defaults may already be near-optimal for this small grid.

**Right parameters?** For a short homework pass, these 2–3 knobs each are **reasonable**: they affect capacity (depth, leaves, learning rate) and **how** the model handles imbalance (RF class weight) or sampling noise (XGB subsample). A “full” tune would add more estimators, `min_child_weight`, colsample, etc. Re-run the cell above and plug your printed numbers into the next assignment sentence: e.g. “Recall went **up/down** by …; I think **yes/no** these were the right levers because …”

## Full classification reports — tuned models

Run the **tuning** cell first so `rf_tuned` and `xgb_tuned` exist. Same layout as the default-model reports above.

In [15]:
labels = [0, 1]
target_names = ["no_default (0)", "default (1)"]

print("=== Tuned Random Forest — train ===")
print(classification_report(y_train, rf_tuned.predict(X_train), labels=labels, target_names=target_names, digits=4))
print("=== Tuned Random Forest — test ===")
print(classification_report(y_test, rf_tuned.predict(X_test), labels=labels, target_names=target_names, digits=4))

print("=== Tuned XGBoost — train ===")
print(classification_report(y_train, xgb_tuned.predict(X_train), labels=labels, target_names=target_names, digits=4))
print("=== Tuned XGBoost — test ===")
print(classification_report(y_test, xgb_tuned.predict(X_test), labels=labels, target_names=target_names, digits=4))

=== Tuned Random Forest — train ===
                precision    recall  f1-score   support

no_default (0)     0.9012    0.8824    0.8917     14953
   default (1)     0.6144    0.6595    0.6362      4247

      accuracy                         0.8331     19200
     macro avg     0.7578    0.7710    0.7639     19200
  weighted avg     0.8378    0.8331    0.8352     19200

=== Tuned Random Forest — test ===
                precision    recall  f1-score   support

no_default (0)     0.8760    0.8561    0.8659      3738
   default (1)     0.5310    0.5734    0.5514      1062

      accuracy                         0.7935      4800
     macro avg     0.7035    0.7148    0.7086      4800
  weighted avg     0.7997    0.7935    0.7963      4800

=== Tuned XGBoost — train ===
                precision    recall  f1-score   support

no_default (0)     0.8620    0.9675    0.9117     14953
   default (1)     0.7989    0.4547    0.5795      4247

      accuracy                         0.8541     1

### Defaults — which was better?

**ROC-AUC (test and CV):** Random Forest edges XGBoost slightly; CV mean is a bit higher and fold-to-fold std is usually lower. **Recall / F1 on defaults (class 1):** XGBoost is often a hair better at the default 0.5 threshold. So “better out of the box” = **RF if you care about AUC**, **XGB if you care more about catching defaults**—plug your printed numbers to confirm.